# AlgoPay SDK demo (Google Colab)

**Package:** [`algopay-sdk` 0.1.0a3 on PyPI](https://pypi.org/project/algopay-sdk/)  
**Import:** `from algopay import AlgoPay`

This notebook is structured for a **voiceover recording**: each section has a short *"Say this"* line you can read aloud.

### What you will showcase

1. Testnet client setup  
2. Agent **wallet** + USDC balance  
3. **Routing** — Algorand address vs `https://` (x402)  
4. **Guards** — policy blocks before chain spend  
5. **`simulate`** — dry-run  
6. **Ledger** — audit trail  
7. **Live testnet** (optional) — USDC transfer + x402 + explorer link  
8. **Payment intents** — authorize → confirm / cancel  
9. **`batch_pay`** — multiple payments  

> **Runtime:** Use **Python 3.10+**. Colab supports top-level `await` in code cells.

> **Live section:** Needs testnet **ALGO** (fees) + **USDC**. See the *Fund your demo wallet* section. Sections 1–6 run **without** funding.

## Install the published SDK

*Say this:* "We install AlgoPay from PyPI — the distribution is `algopay-sdk`, the Python import is `algopay`."

**Install:** `pip install "algopay-sdk==0.1.0a3"` — use **0.1.0a3 or newer**. PyPI **0.1.0a2** shipped an **empty wheel** (no `algopay` module), which caused `ModuleNotFoundError` in Colab and elsewhere. **0.1.0a3** fixes the package build and drops unused `cryptography` / hard `redis` deps (Redis is optional: `pip install "algopay-sdk[redis]"`).

In [ ]:
import subprocess
import sys

SDK_VERSION = "0.1.0a3"

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", f"algopay-sdk=={SDK_VERSION}"]
)

import algopay
from algopay import AlgoPay

assert hasattr(algopay, "AlgoPay"), "algopay package missing from wheel — pin algopay-sdk>=0.1.0a3"
print("algopay version:", getattr(algopay, "__version__", "?"))
print("python:", sys.executable)

## Imports & helpers

In [2]:
from __future__ import annotations

import json
import os
from decimal import Decimal

from algosdk import account as algo_account
from algopay import AlgoPay
from algopay.core.types import Network, PaymentRequest, PaymentStatus

TESTNET_EXPLORER = "https://testnet.explorer.perawallet.app/tx"
X402_DEMO_URL = os.environ.get(
    "ALGOPAY_X402_URL",
    "https://x402.goplausible.xyz/examples/weather",
)


def explorer_link(tx_id: str | None) -> str | None:
    if not tx_id:
        return None
    return f"{TESTNET_EXPLORER}/{tx_id}"


def show_payment_result(result, title: str = "Payment") -> None:
    print(f"--- {title} ---")
    print("success:", result.success)
    print("status:", result.status)
    print("method:", result.method)
    print("amount:", result.amount, "USDC")
    print("recipient:", result.recipient[:80] + ("..." if len(result.recipient) > 80 else ""))
    if result.blockchain_tx:
        print("blockchain_tx:", result.blockchain_tx)
        print("explorer:", explorer_link(result.blockchain_tx))
    if result.error:
        print("error:", result.error)
    if result.guards_passed:
        print("guards_passed:", result.guards_passed)
    if result.resource_data is not None:
        preview = result.resource_data
        if isinstance(preview, dict):
            preview = {k: preview[k] for k in list(preview)[:8]}
        text = str(preview)
        print("resource_data:", text[:500] + ("..." if len(text) > 500 else ""))


print("AlgoPay SDK ready.")
print("Testnet USDC ASA id:", Network.ALGORAND_TESTNET.usdc_asa_id())

ModuleNotFoundError: No module named 'algopay'

## 1. Initialize client (Algorand testnet)

*Say this:* "AlgoPay defaults to Algorand testnet and public Algod/Indexer endpoints — no API keys required for chain reads."

In [ ]:
client = AlgoPay(network=Network.ALGORAND_TESTNET)

print("network:", client.config.network.value)
print("algod:", client.config.algod_url)
print("indexer:", client.config.indexer_url)
print("usdc_asa_id:", client.config.usdc_asa_id)

## 2. Create an agent wallet

*Say this:* "Each agent gets a wallet set and a wallet — a local Algorand keypair. Keys stay in this process unless you plug in your own storage or use the hosted console vault."

> Colab sessions are **ephemeral**. Re-run this cell after a disconnect, or restore a funded wallet in the optional cell below.

In [ ]:
wallet_set = await client.create_wallet_set("colab-demo")
wallet = await client.create_wallet(wallet_set_id=wallet_set.id, name="colab-agent")

WALLET_ID = wallet.id
WALLET_SET_ID = wallet_set.id
WALLET_ADDRESS = wallet.address

print("wallet_set_id:", WALLET_SET_ID)
print("wallet_id:", WALLET_ID)
print("address (fund for live demo):", WALLET_ADDRESS)
print("\nFund testnet ALGO:", "https://bank.testnet.algorand.network/")

### Optional: restore a funded wallet (Colab Secrets)

If you already funded a wallet elsewhere, store these in **Secrets** (🔑 in Colab sidebar):

| Secret | Description |
| ------ | ----------- |
| `ALGOPAY_DEMO_PRIVATE_KEY_B64` | Base64-encoded 64-byte Algorand private key |
| `ALGOPAY_DEMO_WALLET_ID` | Wallet UUID from your prior run |
| `ALGOPAY_DEMO_WALLET_SET_ID` | Wallet set UUID |
| `ALGOPAY_DEMO_ADDRESS` | 58-character address |

Then run the next cell **instead of** re-creating a new wallet (skip the create cell above if restoring).

In [ ]:
import base64

try:
    from google.colab import userdata  # type: ignore

    sk_b64 = userdata.get("ALGOPAY_DEMO_PRIVATE_KEY_B64")
    WALLET_ID = userdata.get("ALGOPAY_DEMO_WALLET_ID")
    WALLET_SET_ID = userdata.get("ALGOPAY_DEMO_WALLET_SET_ID")
    WALLET_ADDRESS = userdata.get("ALGOPAY_DEMO_ADDRESS")
    sk = base64.b64decode(sk_b64)
    if len(sk) != 64:
        raise ValueError("private key must be 64 bytes")
    client.wallet.repository.register_wallet(
        wallet_set_id=WALLET_SET_ID,
        wallet_id=WALLET_ID,
        address=WALLET_ADDRESS,
        private_key=sk,
        network_caip2=Network.ALGORAND_TESTNET.to_caip2(),
        name="colab-restored",
    )
    print("Restored wallet:", WALLET_ADDRESS)
except Exception as e:
    print("Skip restore (not in Colab or secrets missing):", e)
    print("Use the create-wallet cell above instead.")

## 3. Balance & wallet listing

*Say this:* "Balance is USDC on the testnet ASA — not ALGO. You need ALGO separately for transaction fees."

In [ ]:
balance = await client.get_balance(WALLET_ID)
print("USDC balance:", balance)

w = await client.get_wallet(WALLET_ID)
sets = await client.list_wallet_sets()
wallets = await client.list_wallets(WALLET_SET_ID)
print("wallet sets:", len(sets), "| wallets in set:", len(wallets))
print("this wallet:", w.address)

## 4. Payment routing (`can_pay` / `detect_method`)

*Say this:* "One `pay()` API routes by recipient string — a 58-character Algorand address uses a USDC transfer; an HTTPS URL uses x402 pay-per-call."

In [ ]:
_, sample_addr = algo_account.generate_account()

for label, recipient in [
    ("Algorand address", sample_addr),
    ("x402 URL", X402_DEMO_URL),
    ("invalid", "not-a-valid-recipient"),
]:
    print(f"{label!r}: can_pay={client.can_pay(recipient)!r}, method={client.detect_method(recipient)!r}")

## 5. Spending guards (blocked payment — no chain tx)

*Say this:* "Guards run before we touch the chain. Here a recipient allowlist blocks a payment to an unknown address — it shows up as BLOCKED in the result and ledger, not as a failed on-chain transaction."

In [ ]:
# Clean up prior demo guards if re-running
for name in ("colab_whitelist", "colab_budget", "colab_single_tx"):
    try:
        await client.guards.remove_guard(WALLET_ID, name)
    except Exception:
        pass

_, allowed_addr = algo_account.generate_account()
_, blocked_addr = algo_account.generate_account()

await client.add_recipient_guard(
    WALLET_ID,
    mode="whitelist",
    addresses=[allowed_addr],
    name="colab_whitelist",
)
await client.add_budget_guard(WALLET_ID, daily_limit="500", name="colab_budget")
await client.add_single_tx_guard(WALLET_ID, max_amount="100", name="colab_single_tx")

print("guards on wallet:", await client.list_guards(WALLET_ID))

blocked = await client.pay(
    WALLET_ID,
    blocked_addr,
    "0.01",
    purpose="colab-guard-demo",
)
show_payment_result(blocked, "Blocked by recipient guard")
assert blocked.status == PaymentStatus.BLOCKED
assert not blocked.success

## 6. Simulate (dry-run)

*Say this:* "Agents can call `simulate` before spending — guards, balance, and routing — without submitting a transaction."

In [ ]:
sim_transfer = await client.simulate(WALLET_ID, WALLET_ADDRESS, "0.01")
print("simulate self-transfer:", sim_transfer.would_succeed, "|", sim_transfer.reason)
print("route:", sim_transfer.route)

sim_x402 = await client.simulate(WALLET_ID, X402_DEMO_URL, "0.5")
print("simulate x402:", sim_x402.would_succeed, "|", sim_x402.reason)

## 7. Ledger (audit trail)

*Say this:* "Every `pay()` writes a ledger entry — pending, completed, failed, or blocked — for operator audit and agent memory."

In [ ]:
entries = await client.ledger.query(wallet_id=WALLET_ID, limit=5)
print(f"Latest {len(entries)} ledger entries:")
for e in entries:
    print(
        " -",
        e.id[:8] + "...",
        e.status.value,
        e.amount,
        "USDC →",
        (e.recipient[:40] + "..." if len(e.recipient) > 40 else e.recipient),
    )
    if e.tx_hash:
        print("   tx:", e.tx_hash, "|", explorer_link(e.tx_hash))

---

## 8. Live testnet — fund & opt-in

*Say this:* "For a real on-chain demo, fund the address printed above with testnet ALGO, opt in to USDC once, then acquire testnet USDC on ASA `10458941`."

| Step | Link / action |
| ---- | ------------- |
| ALGO (fees) | [TestNet dispenser](https://bank.testnet.algorand.network/) |
| Opt-in USDC | Run the cell below (`RUN_OPT_IN = True`) |
| USDC | Your org faucet / test tap for ASA **10458941** |

Set `RUN_LIVE = True` in the next cell when funded (~0.05 USDC is enough).

In [ ]:
# Toggle these when you are ready to record the on-chain segment
RUN_LIVE = False   # ← set True after funding ALGO + USDC
RUN_OPT_IN = False  # ← set True once to opt in (needs ALGO for fee)

if RUN_OPT_IN:
    opt_tx = client.wallet.opt_in_usdc(WALLET_ID)
    print("USDC opt-in submitted:", opt_tx)
    print("explorer:", explorer_link(opt_tx))
    print("Wait ~30s, fund USDC, then set RUN_LIVE = True")
else:
    print("Opt-in skipped. Address to fund:", WALLET_ADDRESS)

balance = await client.get_balance(WALLET_ID)
print("Current USDC balance:", balance)
if RUN_LIVE and balance < Decimal("0.03"):
    print("WARNING: low USDC — live cells may fail.")

## 9. Live — USDC transfer + explorer link

*Say this:* "`pay` with an Algorand address submits a USDC asset transfer. We use a small self-payment so the receiver is already opted in."

In [ ]:
if not RUN_LIVE:
    print("Set RUN_LIVE = True in the previous cell to execute a real transfer.")
else:
    transfer = await client.pay(
        WALLET_ID,
        WALLET_ADDRESS,
        "0.02",
        purpose="colab-live-transfer",
        skip_guards=True,
        wait_for_completion=True,
        timeout_seconds=180.0,
    )
    show_payment_result(transfer, "Live USDC transfer")
    LAST_LEDGER_FOR_SYNC = None
    rows = await client.ledger.query(wallet_id=WALLET_ID, limit=3)
    if rows and rows[0].tx_hash:
        LAST_LEDGER_FOR_SYNC = rows[0].id
        synced = await client.sync_transaction(LAST_LEDGER_FOR_SYNC)
        print("sync_transaction status:", synced.status.value, "tx:", synced.tx_hash)

## 10. Live — x402 pay-per-call

*Say this:* "The same `pay()` accepts an HTTPS URL. AlgoPay runs the Algorand x402 exact scheme — discover, pay in USDC, fetch the resource."

In [ ]:
if not RUN_LIVE:
    print("Set RUN_LIVE = True to run x402.")
else:
    x402 = await client.pay(
        WALLET_ID,
        X402_DEMO_URL,
        "0.5",
        skip_guards=True,
    )
    show_payment_result(x402, "Live x402 payment")

## 11. Payment intents (authorize → confirm / cancel)

*Say this:* "Intents separate authorization from capture — like holding funds before you confirm the purchase. Confirm runs the same routing as `pay`."

> **Confirm** requires sufficient USDC (`RUN_LIVE = True`). **Cancel** works without funding.

In [ ]:
intent = await client.create_payment_intent(
    WALLET_ID,
    WALLET_ADDRESS,
    "0.02",
    purpose="colab-intent",
)
print("created intent:", intent.id, "status:", intent.status.value)

if RUN_LIVE:
    confirmed = await client.confirm_payment_intent(intent.id)
    show_payment_result(confirmed, "Confirmed intent")
else:
    print("Skipping confirm (set RUN_LIVE = True).")

intent2 = await client.create_payment_intent(WALLET_ID, WALLET_ADDRESS, "0.01")
canceled = await client.cancel_payment_intent(intent2.id)
print("canceled intent:", canceled.id, "status:", canceled.status.value)

## 12. Batch payments

*Say this:* "`batch_pay` runs multiple payment requests with a concurrency limit — useful when an agent settles several invoices in one turn."

In [ ]:
if not RUN_LIVE:
    print("Set RUN_LIVE = True to run batch_pay on chain.")
else:
    batch = await client.batch_pay(
        [
            PaymentRequest(wallet_id=WALLET_ID, recipient=WALLET_ADDRESS, amount=Decimal("0.01")),
            PaymentRequest(wallet_id=WALLET_ID, recipient=X402_DEMO_URL, amount=Decimal("0.5")),
        ],
        concurrency=2,
    )
    print("batch:", batch.success_count, "ok /", batch.failed_count, "failed /", batch.total_count, "total")
    for i, r in enumerate(batch.results):
        print(f"  [{i}] success={r.success} status={r.status} tx={r.blockchain_tx}")

## 13. Indexer transaction history

*Say this:* "`list_transactions` reads recent on-chain activity for the wallet from the Algorand indexer."

In [ ]:
txs = await client.list_transactions(WALLET_ID)
print(f"Indexer returned {len(txs)} transaction(s)")
for t in txs[:5]:
    print(" -", t.id, t.state, explorer_link(t.id))

## 14. Save wallet ids for your next session (optional)

Copy these into Colab **Secrets** after your first run so you can restore a funded wallet later.

In [ ]:
rec = client.wallet.repository.get_wallet(WALLET_ID)
if rec:
    import base64

    payload = {
        "ALGOPAY_DEMO_WALLET_ID": WALLET_ID,
        "ALGOPAY_DEMO_WALLET_SET_ID": WALLET_SET_ID,
        "ALGOPAY_DEMO_ADDRESS": WALLET_ADDRESS,
        "ALGOPAY_DEMO_PRIVATE_KEY_B64": base64.b64encode(rec.private_key).decode("ascii"),
    }
    print(json.dumps({k: v for k, v in payload.items() if k != "ALGOPAY_DEMO_PRIVATE_KEY_B64"}, indent=2))
    print("\n(private key omitted from print — copy from repository or store sk in Secrets only)")
    print("Store ALGOPAY_DEMO_PRIVATE_KEY_B64 in Secrets; never commit it.")

---

## Recording checklist (voiceover)

| Segment | Cells | Needs funding? |
| ------- | ----- | ---------------- |
| Intro + install | top | No |
| Wallet + routing + guards + simulate + ledger | 1–7 | No |
| **Money shot** — transfer explorer link | 9 | **Yes** |
| x402 + resource JSON | 10 | **Yes** |
| Intents + batch | 11–12 | Confirm/batch: **Yes** |
| Wrap-up | 13–14 | No |

**Links for the outro:**

- PyPI: https://pypi.org/project/algopay-sdk/ (use `algopay-sdk>=0.1.0a3`)
- Docs: https://algodev-studio.github.io/algopay-sdk/
- GitHub: https://github.com/Algodev-Studio/algopay-sdk